In [10]:
# test_ml_vmc.py
from deepqmc.molecule import Molecule
from deepqmc.hamil import MolecularHamiltonian
from deepqmc.train import train
from deepqmc.sampling import combine_samplers, MetropolisSampler, DecorrSampler
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
import deepqmc
from deepqmc.app import instantiate_ansatz
from deepqmc.train import train
from functools import partial
import os

# Create H2 molecule
mol = Molecule(
    coords=[[0.0, 0.0, 0.0], [0.0, 0.0, 0.74]], 
    charges=[1, 1], charge=0, spin=0, unit='angstrom'
)

# Create Hamiltonian
H = MolecularHamiltonian(mol=mol)

# Load default neural ansatz
deepqmc_dir = os.path.dirname(deepqmc.__file__)
config_dir = os.path.join(deepqmc_dir, 'conf/ansatz')
with initialize_config_dir(version_base=None, config_dir=config_dir):
    cfg = compose(config_name='default')
ansatz = instantiate_ansatz(H, instantiate(cfg))

def sampler_factory(hamil, wf):
    return combine_samplers([
        DecorrSampler(length=10),
        MetropolisSampler(hamil, wf)
    ], hamil, wf)
    

# Load optimizer
opt_config_dir = os.path.join(deepqmc_dir, 'conf/task/opt')
with initialize_config_dir(version_base=None, config_dir=opt_config_dir):
    opt_cfg = compose(config_name='kfac')
optimizer = instantiate(opt_cfg)

# Train
train(H, ansatz, optimizer, sampler_factory, 
      steps=1000, electron_batch_size=500, seed=42, workdir='h2_test')


TypeError: sampler_factory() takes 2 positional arguments but 6 were given